# RunPod FLUX.1-dev LoRA Training Notebook

This notebook provides a step-by-step guide for training FLUX.1-dev LoRA models on RunPod.

## Prerequisites
- RunPod instance with at least 24GB VRAM
- Hugging Face account with READ token
- Dataset of images (4-30 images recommended)

## 📋 Quick Reference
- **Template:** `runpod/pytorch:2.2.0-py3.10-cuda12.1.1-devel-ubuntu22.04`
- **Min GPU:** 24GB VRAM (A40, A100, RTX 4090)
- **Storage:** 120GB+ recommended
- **Training Time:** 1-4 hours typical

## Step 1: Environment Setup

First, let's check our GPU and set up the environment:

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Run the automated setup script
!bash scripts/setup_runpod.sh

## Step 2: Hugging Face Authentication

You need to authenticate with Hugging Face to access FLUX.1-dev:

In [ ]:
# Login to Hugging Face
# You'll need a READ token from https://huggingface.co/settings/tokens
# Make sure to accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
!huggingface-cli login

## Step 3: Dataset Upload

Upload your training images to the dataset folder:

In [ ]:
# Create dataset directory if it doesn't exist
import os
os.makedirs('/workspace/ai-toolkit/dataset', exist_ok=True)

# List current contents
!ls -la /workspace/ai-toolkit/dataset/

**Manual Step:** Upload your images to the dataset folder using:
- RunPod's file manager interface
- Jupyter's file upload feature
- Command line tools like `wget` or `curl`

Each image should have a corresponding `.txt` file with the same name containing the caption.

In [ ]:
# Example: Create caption files (modify as needed)
# Replace with your actual image filenames and captions

captions = {
    'image1.jpg': 'a person in a red jacket standing outdoors',
    'image2.jpg': 'a person wearing sunglasses in a park',
    'image3.jpg': 'portrait of a person with dramatic lighting'
}

for image_name, caption in captions.items():
    caption_file = image_name.replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
    with open(f'/workspace/ai-toolkit/dataset/{caption_file}', 'w') as f:
        f.write(caption)
    print(f"Created caption file: {caption_file}")

## Step 4: Training Configuration

Let's set up the training configuration:

In [ ]:
# Copy the RunPod-optimized config template
!cp config/examples/train_lora_flux_runpod.yaml config/my_flux_training.yaml

# Show the config file
!cat config/my_flux_training.yaml

## Step 5: Customize Configuration

Edit the configuration file to match your needs:

In [ ]:
# Edit configuration parameters
import yaml

# Load config
with open('config/my_flux_training.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Customize these values:
config['config']['name'] = 'my_runpod_flux_lora'  # Your model name
config['config']['process'][0]['datasets'][0]['folder_path'] = '/workspace/ai-toolkit/dataset'
config['config']['process'][0]['train']['steps'] = 1000  # Adjust based on your dataset

# Optional: Add trigger word
# config['config']['process'][0]['trigger_word'] = 'your_trigger_word'

# Optional: Enable Hugging Face upload
# config['config']['process'][0]['save']['push_to_hub'] = True
# config['config']['process'][0]['save']['hf_repo_id'] = 'your-username/your-model-name'

# Save updated config
with open('config/my_flux_training.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Configuration updated successfully!")

## Step 6: Start Training

Now let's start the training process:

In [ ]:
# Verify dataset before training
!echo "Dataset contents:"
!ls -la /workspace/ai-toolkit/dataset/

# Count images and captions
import os
dataset_path = '/workspace/ai-toolkit/dataset'
images = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
captions = [f for f in os.listdir(dataset_path) if f.endswith('.txt')]

print(f"\nImages found: {len(images)}")
print(f"Captions found: {len(captions)}")

if len(images) != len(captions):
    print("⚠️  Warning: Number of images and captions don't match!")
else:
    print("✅ Dataset looks good!")

In [ ]:
# Start training
# This will take 1-4 hours depending on your settings
!python run.py config/my_flux_training.yaml

## Step 7: Monitor Training Progress

While training is running, you can monitor the progress:

In [ ]:
# Check training outputs
!ls -la output/

In [ ]:
# View sample images (if generated)
import os
from IPython.display import Image, display

sample_dir = 'output/my_runpod_flux_lora/samples'
if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    for sample_file in sample_files[:5]:  # Show first 5 samples
        print(f"Sample: {sample_file}")
        display(Image(os.path.join(sample_dir, sample_file)))
else:
    print("No samples generated yet. Training may still be in progress.")

## Step 8: Access Your Trained Model

Once training is complete, your model will be available:

In [ ]:
# List training outputs
!ls -la output/my_runpod_flux_lora/

# Show model file sizes
!du -h output/my_runpod_flux_lora/*.safetensors

## Step 9: Alternative Training Methods

You can also use the Gradio UI for a more interactive experience:

In [ ]:
# Start the Gradio UI
# This will create a web interface for training
!python flux_train_ui.py

## 🎯 Summary

This notebook walked you through:

1. ✅ Setting up the RunPod environment
2. ✅ Authenticating with Hugging Face
3. ✅ Preparing your dataset
4. ✅ Configuring training parameters
5. ✅ Starting and monitoring training
6. ✅ Accessing your trained model

## 📚 Additional Resources

- **Detailed Guide:** [docs/RUNPOD_FLUX_TRAINING_GUIDE.md](docs/RUNPOD_FLUX_TRAINING_GUIDE.md)
- **AI Toolkit Repository:** [https://github.com/ostris/ai-toolkit](https://github.com/ostris/ai-toolkit)
- **FLUX.1-dev Model:** [https://huggingface.co/black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)

## 🔧 Troubleshooting

Common issues and solutions:

- **CUDA Out of Memory:** Enable `low_vram: true` in config
- **Permission Errors:** Run `chmod -R 755 /workspace/ai-toolkit/`
- **Model Download Issues:** Clear cache with `rm -rf ~/.cache/huggingface/`
- **Training Stuck:** Check GPU usage with `nvidia-smi`

Happy training! 🚀